In [ ]:


# ================== 0) Mount & Imports ==================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random, pickle, zipfile
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

GN_GROUPS = 32

def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

# ================== 1) ResNet18 Backbone + Single Head ==================
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(
        in_planes, out_planes,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False
    )

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()

        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)

        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                make_gn(planes * self.expansion)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()

        block = BasicBlock
        num_blocks = [2,2,2,2]

        self.expansion = block.expansion
        self.nf = nf
        self.in_planes = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)

        self.layer1 = self._make_layer(block, nf,     num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, nf * 2, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, nf * 4, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, nf * 8, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)

        layers = []

        in_planes = self.in_planes

        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion

        self.in_planes = in_planes

        return nn.Sequential(*layers)

    def forward(self, x):

        out = torch.relu(self.gn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = F.avg_pool2d(out, out.shape[2])

        feat = out.view(out.size(0), -1)

        return feat

    @property
    def out_dim(self):
        return self.nf * 8 * self.expansion

class SingleHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone, num_classes=40):
        super().__init__()

        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):

        feat = self.backbone(x)

        logits = self.head(feat)

        return logits

# ================== 2) Paths ==================
BASE = "/content/drive/MyDrive/ML_Project/project_files/QDA_Tiny_Imagenet/CS"
os.makedirs(BASE, exist_ok=True)

ZIP_PATH = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"

DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

MODEL_PATH = os.path.join(
    BASE,
    "finetuned_tinyimg_task5_best_qda_classes80_99.pth"
)


SAVE_TOPK_PATH = os.path.join(
    BASE,
    "CS_QDA_tinyimg_classes80_99_head100_topk.pkl"
)

SAVE_NEIGHBORS_PATH = os.path.join(
    BASE,
    "CS_QDA_neighbors_tinyimg_classes80_99_head100.pkl"
)

TOP_K = 300000

NUM_CS_SAMPLES = 2000

BATCH_SIZE = 32
NUM_WORKERS = 0

# ================== 3) Tiny ImageNet ==================
def ensure_extracted(zip_path: str, data_root: str):

    processed_dir = os.path.join(data_root, "processed")

    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root

    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found:\n{zip_path}")

    os.makedirs(data_root, exist_ok=True)

    print(f"[INFO] Extracting ZIP...\n{zip_path}")

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)

    return data_root

DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):

    def __init__(self, root: str, train: bool = True, transform=None):

        self.root = root
        self.train = train
        self.transform = transform

        split = "train" if train else "val"

        xs, ys = [], []


        for num in range(20):

            xs.append(
                np.load(
                    os.path.join(
                        root,
                        f"processed/x_{split}_{num+1:02d}.npy"
                    )
                )
            )

            ys.append(
                np.load(
                    os.path.join(
                        root,
                        f"processed/y_{split}_{num+1:02d}.npy"
                    )
                )
            )

        self.data = np.concatenate(np.array(xs))

        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        img, target = self.data[index], int(self.targets[index])

        img = Image.fromarray(np.uint8(255 * img))

        if self.transform is not None:
            img = self.transform(img)

        return img, target

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        TIN_IMAGENET_MEAN,
        TIN_IMAGENET_STD
    ),
])

train_full = TinyImagenet(
    DATA_ROOT,
    train=False,
    transform=tf
)

# ================== Classes 20-39 WITHOUT remap ==================
keep_classes = list(range(80, 100))

idx_next20 = [
    i for i, t in enumerate(train_full.targets)
    if int(t) in keep_classes
]

train_next20 = Subset(train_full, idx_next20)

def get_random_subset(dataset, n_samples=2000, seed=42):

    rng = random.Random(seed)

    n = len(dataset)

    k = min(n_samples, n)

    idxs = rng.sample(range(n), k)

    return Subset(dataset, idxs)

subset = get_random_subset(
    train_next20,
    n_samples=NUM_CS_SAMPLES,
    seed=SEED
)

subset_loader = DataLoader(
    subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

# ================== 4) Tools ==================
def unravel_index(index, shape):

    dims = []

    for s in reversed(shape):
        dims.append(index % s)
        index //= s

    return tuple(reversed(dims))

def ravel_index(multi_idx, shape):

    flat = 0

    for idx, dim in zip(multi_idx, shape):
        flat = flat * dim + idx

    return flat

# ================== 5) Confidence Sensitivity ==================
def compute_confidence_sensitivity(model, dataloader, device):

    model.eval()

    cs = {
        n: torch.zeros_like(p, device=device)
        for n, p in model.named_parameters()
        if p.requires_grad and (not n.startswith("head"))
    }

    total_samples = 0

    for inputs, _ in dataloader:

        inputs = inputs.to(device)

        for ex in inputs:

            ex = ex.unsqueeze(0)

            model.zero_grad(set_to_none=True)

            logits = model(ex)

            pred = torch.argmax(logits, dim=1)

            fx = logits.gather(
                1,
                pred.view(1,1)
            ).squeeze()

            fx.backward()

            for name, p in model.named_parameters():

                if name.startswith("head"):
                    continue

                if p.grad is not None:
                    cs[name] += (p.grad.detach() ** 2)

            total_samples += 1

    for name in cs:
        cs[name] /= max(1, total_samples)

    return cs

# ================== 6) Top-K ==================
def get_topk_cs_weights(cs_dict, model_state_dict, k):

    if k <= 0:
        return []

    all_entries = []

    for name, cs_tensor in cs_dict.items():

        flat_cs = cs_tensor.flatten()

        flat_w = model_state_dict[name].flatten()

        for i in range(flat_cs.numel()):

            all_entries.append({
                "name": name,
                "index": i,
                "value": float(flat_w[i].item()),
                "cs": float(flat_cs[i].item())
            })

    all_entries.sort(
        key=lambda x: x["cs"],
        reverse=True
    )

    return all_entries[:k]

# ================== 7) Neighbors ==================
def extract_conv_neighbors(topk_entries, model, cs_dict):

    neighbors = []

    if not topk_entries:
        return neighbors

    param_shapes = {
        name: p.shape
        for name, p in model.named_parameters()
    }

    cs_flat = {
        name: tens.flatten()
        for name, tens in cs_dict.items()
    }

    neighbor_offsets = [
        (0,0,-1,-1),
        (0,0,-1,1),
        (0,0,1,-1),
        (0,0,1,1),

        (0,0,-1,0),
        (0,0,1,0),
        (0,0,0,-1),
        (0,0,0,1),
    ]

    topk_set = set(
        (e["name"], e["index"])
        for e in topk_entries
    )

    added = set()

    for e in topk_entries:

        name, flat_idx = e["name"], e["index"]

        shape = param_shapes[name]

        if len(shape) != 4:
            continue

        oc, ic, kh, kw = unravel_index(flat_idx, shape)

        for do, di, dh, dw in neighbor_offsets:

            no, ni, nh, nw = (
                oc + do,
                ic + di,
                kh + dh,
                kw + dw
            )

            if (
                0 <= no < shape[0]
                and 0 <= ni < shape[1]
                and 0 <= nh < shape[2]
                and 0 <= nw < shape[3]
            ):

                n_flat = ravel_index(
                    (no, ni, nh, nw),
                    shape
                )

                key = (name, n_flat)

                if key in topk_set or key in added:
                    continue

                neighbors.append({
                    "name": name,
                    "index": n_flat,
                    "position": (no, ni, nh, nw),
                    "cs": float(cs_flat[name][n_flat].item())
                })

                added.add(key)

    return neighbors

# ================== 8) Build model ==================
model = SingleHeadNet(
    backbone=ResNet18Backbone(nf=64),
    num_classes=100
).to(DEVICE)

ckpt = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

state = (
    ckpt["state"]
    if isinstance(ckpt, dict) and "state" in ckpt
    else ckpt
)

model.load_state_dict(state, strict=True)

print(f"[INFO] Loaded checkpoint from {MODEL_PATH}")

# ================== 9) Compute CS ==================
cs_info = compute_confidence_sensitivity(
    model,
    subset_loader,
    DEVICE
)


topk_info = get_topk_cs_weights(
    cs_info,
    model.state_dict(),
    TOP_K
)

neighbors_info = extract_conv_neighbors(
    topk_info,
    model,
    cs_info
)

with open(SAVE_TOPK_PATH, "wb") as f:
    pickle.dump(topk_info, f)

with open(SAVE_NEIGHBORS_PATH, "wb") as f:
    pickle.dump(neighbors_info, f)

# ================== 10) Stats ==================
print(
    f"✅ Confidence Sensitivity computed "
    f"from {len(subset)} samples."
)

print(f"✅ Top-K total: {len(topk_info)}")

conv_topk = [
    e for e in topk_info
    if len(model.state_dict()[e['name']].shape) == 4
]

if len(topk_info) > 0:

    print(
        f"📌 Top-K from Conv2D: "
        f"{len(conv_topk)} "
        f"({100 * len(conv_topk) / len(topk_info):.2f}%)"
    )

else:
    print("📌 Top-K from Conv2D: 0")

print(f"📌 Neighbors extracted: {len(neighbors_info)}")

print("\n📊 Confidence Sensitivity Statistics:")

print(
    f"{'Parameter':40s} | "
    f"{'Mean':>12s} | "
    f"{'Min':>12s} | "
    f"{'Max':>12s}"
)

print("-" * 85)

for name, tens in cs_info.items():

    vals = tens.detach().cpu().view(-1)

    mean_val = vals.mean().item()
    min_val = vals.min().item()
    max_val = vals.max().item()

    print(
        f"{name:40s} | "
        f"{mean_val:12.4e} | "
        f"{min_val:12.4e} | "
        f"{max_val:12.4e}"
    )

print("✔️ Done.")

Mounted at /content/drive
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Loaded checkpoint from /content/drive/MyDrive/ML_Project/project_files/QDA_Tiny_Imagenet/CS/finetuned_tinyimg_task5_best_qda_classes80_99.pth
✅ Confidence Sensitivity computed from 1000 samples.
✅ Top-K total: 300000
📌 Top-K from Conv2D: 291465 (97.16%)
📌 Neighbors extracted: 213846

📊 Confidence Sensitivity Statistics:
Parameter                                |         Mean |          Min |          Max
-------------------------------------------------------------------------------------
backbone.conv1.weight                    |   6.8016e-02 |   5.0237e-03 |   4.1973e-01
backbone.gn1.weight                      |   2.1265e-01 |   2.1109e-02 |   7.0435e-01
backbone.gn1.bias                        |   1.5931e-01 |   3.8649e-02 |   3.8877e-01
backbone.layer1.0.conv1.weight           |   1.4983e-03 |   1.8236e-05 |   4.9331e-02
backbone.layer1.0.gn1.weight            